# GS33 — Step 2: Build ConvNeXt-StarDist tile dataset

Builds a balanced 256×256 tile dataset from the GS33 whole-slide images
and CODA-labelled GeoJSONs.

**Pipeline:**
1. Load per-slide cell counts → compute 4-tier sampling weights
2. Build slide manifest (NDPI + GeoJSON pairs)
3. Parse GeoJSON centroids, class labels, **and full polygon features**
4. Generate non-overlapping tile grid, filter low-cell tiles
5. Weighted unique tile selection (~30k tiles)
6. Slide-level train/val split (80/20)
7. Oversample rare classes (train only, augmented `_dup{k}` copies)
8. Extract PNG tiles, per-tile GeoJSONs, uint16 instance masks, and inst2class.json sidecars
9. Write split CSVs and ConvNeXt-StarDist training config

**Output per tile** (under `dataset_256_30k_GS33/{train|val}/`):

| Folder | File | Content |
|--------|------|---------|
| `images/` | `tile_id.png` | RGB image (256×256) |
| `geojsons/` | `tile_id.geojson` | Localized polygon GeoJSON with class labels |
| `labels/` | `tile_id.png` | uint16 instance mask (0 = bg, 1…N = nuclei) |
| `labels/` | `tile_id_inst2class.json` | `{"1": "kidney", "2": "bone", ...}` |

| Cell | Purpose |
|------|---------|
| 1 | Imports |
| **2** | **Parameters ← edit here** |
| 3 | Class distribution + sampling weights |
| 4 | Build manifest |
| 5 | Parse GeoJSON annotations |
| 6 | Generate tile candidates |
| 7 | Weighted tile selection |
| 8 | Train / val split |
| 9 | Oversample rare classes |
| 10 | Extract tiles: PNG + GeoJSON + instance mask + inst2class.json |
| 11 | Write splits and training config |
| 12 | Organ coverage summary |

---
## 1 · Imports

In [3]:
import json
import re
import sys
import yaml
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import openslide
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd()))
from dataset_utils import (
    calculate_hybrid_weights,
    get_slide_mpp,
    assign_cells_to_tiles,
    augment_image,
    augment_coords,
    augment_polygon_ring,
    clip_features_to_tile,
    rasterize_tile_features,
    write_tile_geojson,
    polygon_centroid,
)

print("Imports OK")

Imports OK


---
## 2 · Parameters  ← edit here

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
CELL_COUNTS_CSV = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS33\cellvit_training\cell_type_analysis\cell_counts_per_slide.csv")
GEOJSON_DIR     = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS33\geojson_CODAclass")
# NDPIs live directly in the GS33 folder
NDPI_BASE_DIR   = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS33")
OUT_DIR         = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS33\cellvit_training")
OUT_BASE        = OUT_DIR / "dataset_256_30k_GS33"

# ── Class palette ──────────────────────────────────────────────────────────────
LABELS = [
    "bone",   "brain",  "eye",       "heart",     "lungs",
    "GI",     "liver",  "spleen",    "pancreas",  "kidney",
    "mesokidney", "collagen", "ear", "nontissue", "thymus",
    "thyroid", "bladder", "skull",   "spleen2",
]
COLORS = [
    [214,212,161], [247,184, 67], [136,232, 95], [140, 13, 13], [ 38, 27,166],
    [ 13,125, 11], [179, 50,108], [228,235,131], [156, 96,235], [ 46,190,230],
    [150,255,245], [254,222,255], [235,154,108], [255,255,255], [  9, 64,116],
    [255,255, 74], [178,178,  0], [214,212,161], [ 54, 83, 89],
]

# ── Slide subsampling ─────────────────────────────────────────────────────────
# GS33 has ~220 slides — use 1 every SLIDE_STRIDE to keep a manageable subset
SLIDE_STRIDE    = 5    # 220 slides → ~44 slides used

# ── Tile extraction ────────────────────────────────────────────────────────────
TILE_SIZE       = 256   # px at 20x
STRIDE          = 256   # non-overlapping grid
MIN_CELLS_TILE  = 5     # discard tiles with fewer cells
NONTISSUE_FRAC  = 0.75  # discard tiles where >75% of cells are "nontissue"
TARGET_TILES    = 35000 # unique tiles to sample; ~30k go to train, ~5k to val
SEED            = 1337

# ── Final dataset sizes ────────────────────────────────────────────────────────
TRAIN_FINAL = 30000   # exact training set size (after oversampling + cap)
VAL_TARGET  = 5000    # approximate validation set size

# ── Train / val split ─────────────────────────────────────────────────────────
TRAIN_RATIO = 0.86
VAL_RATIO   = 0.14

name_to_id = {name: i for i, name in enumerate(LABELS)}
id_to_name = {i: name for i, name in enumerate(LABELS)}

print(f"Cell counts CSV : {CELL_COUNTS_CSV.exists()}")
print(f"GeoJSON dir     : {GEOJSON_DIR.exists()}")
print(f"NDPI base dir   : {NDPI_BASE_DIR.exists()}")
print(f"Output base     : {OUT_BASE}")
print(f"Slide stride    : every {SLIDE_STRIDE}th slide → ~{220 // SLIDE_STRIDE} slides used")
print(f"Target dataset  : {TRAIN_FINAL:,} train  +  ~{VAL_TARGET:,} val")

---
## 3 · Class distribution and sampling weights

> **After running notebook 1**, review the GS33 class distribution and adjust
> `DOWNSAMPLE_WEIGHTS` below if the dominant classes differ from GS55.

In [5]:
# ── Load class distribution and compute sampling weights ──────────────────────
df_counts   = pd.read_csv(CELL_COUNTS_CSV, index_col=0)
if "TOTAL" in df_counts.columns:
    df_counts = df_counts.drop(columns=["TOTAL"])

class_totals   = df_counts.sum(axis=0).sort_values(ascending=False)
total_cells    = int(class_totals.sum())
hybrid_weights = calculate_hybrid_weights(class_totals)

print(f"Total cells: {total_cells:,}  |  Classes: {len(class_totals)}")
print(f"\n{'Class':<20} {'Count':>12} {'%':>7} {'Weight':>9}")
print("-" * 55)
for cls, cnt in class_totals.items():
    print(f"{cls:<20} {cnt:>12,} {cnt/total_cells*100:>6.1f}% {hybrid_weights[cls]:>9.3f}")

Total cells: 4,671,635  |  Classes: 16

Class                       Count       %    Weight
-------------------------------------------------------
collagen                2,609,357   55.9%     0.050
brain_+_spinal_cord     1,310,643   28.1%     0.050
bone_skull                270,504    5.8%     0.500
liver                     251,835    5.4%     0.500
gi_track                   62,821    1.3%     0.700
nontissue                  49,490    1.1%     0.889
heart                      47,427    1.0%     0.927
kidney                     26,790    0.6%     1.641
eye                        17,759    0.4%     2.476
ear                        13,527    0.3%     3.251
lungs                       5,243    0.1%     8.387
thyroid                     3,026    0.1%    10.000
bladder                     2,715    0.1%    10.000
spleen                        260    0.0%    10.000
thymus                        150    0.0%    10.000
pancreas                       88    0.0%    10.000


---
## 4 · Build slide manifest

In [6]:
# ── Build slide manifest (GeoJSON + NDPI path pairs) ─────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)

gj_files_all = sorted(GEOJSON_DIR.glob("*.geojson"))
gj_files     = gj_files_all[::SLIDE_STRIDE]   # subsample: 1 every SLIDE_STRIDE
print(f"Slide subsampling: 1 every {SLIDE_STRIDE} → {len(gj_files)} / {len(gj_files_all)} slides used")

manifest = []
no_ndpi  = []

for gj_path in gj_files:
    ndpi_path = NDPI_BASE_DIR / f"{gj_path.stem}.ndpi"
    if ndpi_path.exists():
        manifest.append({
            "slide_id":     gj_path.stem,
            "image_path":   str(ndpi_path),
            "geojson_path": str(gj_path),
        })
    else:
        no_ndpi.append(gj_path.stem)

manifest_path = OUT_DIR / "GS33_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Matched   : {len(manifest)} / {len(gj_files)} slides")
print(f"No NDPI   : {len(no_ndpi)}{' — check NDPI_BASE_DIR' if no_ndpi else ''}")
print(f"Manifest  → {manifest_path}")
if no_ndpi[:5]:
    print("  Missing:", no_ndpi[:5])

Slide subsampling: 1 every 10 → 22 / 220 slides used
Matched   : 22 / 22 slides
No NDPI   : 0
Manifest  → \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS33\cellvit_training\GS33_manifest.json


---
## 5 · Parse GeoJSON annotations

In [7]:
# ── Parse GeoJSON annotations → per-slide cell arrays ─────────────────────────
# slide_data[slide_id] stores:
#   cells_xy     : (N, 2) float32 centroids in slide pixel space
#   cells_labels : (N,)   int32   class indices
#   features     : list of N GeoJSON feature dicts (same order as cells_xy)
slide_data = {}

for entry in tqdm(manifest, desc="Parsing GeoJSONs"):
    gj_path   = Path(entry["geojson_path"])
    raw       = json.loads(gj_path.read_text(encoding="utf-8"))
    all_feats = raw if isinstance(raw, list) else raw.get("features", [])

    centroids, labels, features = [], [], []
    for feat in all_feats:
        class_name = feat.get("properties", {}).get("classification", {}).get("name")
        if not class_name or class_name not in name_to_id:
            continue
        geom = feat.get("geometry", {})
        if geom.get("type") != "Polygon":
            continue
        ring = geom.get("coordinates", [[]])[0]
        cx, cy = polygon_centroid(ring)
        if cx is None:
            continue
        centroids.append([cx, cy])
        labels.append(name_to_id[class_name])
        features.append(feat)

    if centroids:
        slide_data[entry["slide_id"]] = {
            "image_path":   entry["image_path"],
            "cells_xy":     np.array(centroids, dtype=np.float32),
            "cells_labels": np.array(labels,    dtype=np.int32),
            "features":     features,
        }

total_parsed = sum(len(v["cells_xy"]) for v in slide_data.values())
print(f"Parsed {len(slide_data)} slides, {total_parsed:,} cells")

Parsing GeoJSONs:   0%|          | 0/22 [00:00<?, ?it/s]

Parsed 22 slides, 3,027,667 cells


---
## 6 · Generate tile candidates

In [8]:
# ── Generate tile candidates ──────────────────────────────────────────────────
NONTISSUE_ID = name_to_id.get("nontissue", -1)

all_tiles = []
for slide_id, data in tqdm(slide_data.items(), desc="Building tile grid"):
    cells_xy     = data["cells_xy"]
    cells_labels = data["cells_labels"]
    tile_groups  = assign_cells_to_tiles(cells_xy, TILE_SIZE, STRIDE)

    for (tx, ty), cell_idxs in tile_groups.items():
        if len(cell_idxs) < MIN_CELLS_TILE:
            continue
        nt_frac = sum(1 for ci in cell_idxs if cells_labels[ci] == NONTISSUE_ID) / len(cell_idxs)
        if nt_frac > NONTISSUE_FRAC:
            continue
        class_counts = Counter(cells_labels[cell_idxs].tolist())
        all_tiles.append({
            "slide_id":     slide_id,
            "tile_x":       tx,
            "tile_y":       ty,
            "cell_indices": cell_idxs,
            "num_cells":    len(cell_idxs),
            "class_counts": class_counts,
        })

print(f"Candidate tiles: {len(all_tiles):,}")

Building tile grid:   0%|          | 0/22 [00:00<?, ?it/s]

Candidate tiles: 48,916


---
## 7 · Weighted unique tile selection

> Adjust `DOWNSAMPLE_WEIGHTS` after reviewing the GS33 class distribution in notebook 1.
> Classes not listed here use `OTHER_WEIGHT = 1.0`; globally rare classes (<1%) get `RARE_WEIGHT = 20.0`.

In [9]:
# ── Weighted unique tile selection ────────────────────────────────────────────
# Per-class downsampling weights — lower = fewer tiles selected from that class.
# *** Review and adjust after running notebook 1 for GS33-specific distribution ***
DOWNSAMPLE_WEIGHTS = {
    "collagen":  0.002,   # extremely dominant in GS55; adjust if GS33 differs
    "nontissue": 0.008,
    "skull":     0.015,
    "brain":     0.025,
    "bone":      0.04,
    "liver":     0.04,
    "GI":        0.04,
}
RARE_WEIGHT  = 20.0
OTHER_WEIGHT =  1.0

RARE_IDS = {
    name_to_id[n] for n in LABELS
    if n in class_totals.index and class_totals[n] / total_cells < 0.01
}

weights = []
valid   = []
for idx, tile in enumerate(all_tiles):
    tc = tile["num_cells"]
    if tc == 0:
        continue
    value = 0.0
    for cls_id, cnt in tile["class_counts"].items():
        cname = id_to_name.get(cls_id, "")
        if cname in DOWNSAMPLE_WEIGHTS:
            value += cnt * DOWNSAMPLE_WEIGHTS[cname]
        elif cls_id in RARE_IDS:
            value += cnt * RARE_WEIGHT
        else:
            value += cnt * OTHER_WEIGHT
    value /= tc
    dom_cnt = sum(
        tile["class_counts"].get(name_to_id[n], 0)
        for n in DOWNSAMPLE_WEIGHTS if name_to_id.get(n) in tile["class_counts"]
    )
    if tc > 0:
        dom_frac = dom_cnt / tc
        if dom_frac > 0.7:   value *= 0.01
        elif dom_frac > 0.5: value *= 0.05
    valid.append(idx)
    weights.append(value)

weights = np.array(weights, dtype=float)
probs   = np.maximum(weights, 1e-10)
probs  /= probs.sum()

np.random.seed(SEED)
n_sample       = min(TARGET_TILES, len(valid))
sel_indices    = np.random.choice(valid, size=n_sample, replace=False, p=probs)
selected_tiles = [all_tiles[i] for i in sel_indices]

print(f"Selected {len(selected_tiles):,} unique tiles (target: {TARGET_TILES:,})")

sampled_cc = Counter()
for tile in selected_tiles:
    sampled_cc.update(tile["class_counts"])

print(f"\n{'Class':<18} {'Original':>10} {'Sampled':>10} {'Ratio':>8}")
print("-" * 50)
for cname in LABELS:
    cid  = name_to_id[cname]
    orig = class_totals.get(cname, 0)
    samp = sampled_cc.get(cid, 0)
    print(f"{cname:<18} {orig:>10,} {samp:>10,} {samp/orig if orig else 0:>7.2f}x")

Selected 35,000 unique tiles (target: 35,000)

Class                Original    Sampled    Ratio
--------------------------------------------------
bone                        0          0    0.00x
brain                       0          0    0.00x
eye                    17,759     17,752    1.00x
heart                  47,427     47,230    1.00x
lungs                   5,243      5,239    1.00x
GI                          0          0    0.00x
liver                 251,835    251,743    1.00x
spleen                    260        260    1.00x
pancreas                   88         88    1.00x
kidney                 26,790     26,790    1.00x
mesokidney                  0          0    0.00x
collagen            2,609,357  1,759,485    0.67x
ear                    13,527     13,512    1.00x
nontissue              49,490     22,957    0.46x
thymus                    150        150    1.00x
thyroid                 3,026      3,026    1.00x
bladder                 2,715      2,690    0.99x
sk

---
## 8 · Train / val split (80 / 20)

In [10]:
# ── Slide-level train / val split ────────────────────────────────────────────
tiles_by_slide = defaultdict(list)
for tile in selected_tiles:
    tiles_by_slide[tile["slide_id"]].append(tile)

slide_ids   = list(tiles_by_slide.keys())
total_count = len(selected_tiles)
n_slides    = len(slide_ids)

max_val_slides = max(2, round(n_slides * VAL_RATIO))

class_to_slides = defaultdict(list)
for sid, tiles in tiles_by_slide.items():
    for tile in tiles:
        for cls_id in tile["class_counts"]:
            class_to_slides[cls_id].append(sid)
class_to_slides = {k: list(set(v)) for k, v in class_to_slides.items()}

BOOST_CLASSES = {"lungs", "spleen", "pancreas", "heart", "eye", "kidney", "bladder", "thyroid"}
BOOST_IDS     = {name_to_id[n] for n in BOOST_CLASSES if n in name_to_id}
priority_ids  = RARE_IDS | BOOST_IDS

np.random.seed(SEED)
train_slides, val_slides = set(), set()

for cls_id in sorted(priority_ids):
    if len(val_slides) >= max_val_slides:
        break
    candidates = [
        s for s in class_to_slides.get(cls_id, [])
        if s not in train_slides and s not in val_slides
    ]
    if candidates:
        candidates.sort(key=lambda s: len(tiles_by_slide[s]), reverse=True)
        val_slides.add(candidates[0])

target_train_tiles = int(total_count * TRAIN_RATIO)
remaining = sorted(
    [s for s in slide_ids if s not in train_slides and s not in val_slides],
    key=lambda s: len(tiles_by_slide[s]), reverse=True,
)
for sid in remaining:
    current_train = sum(len(tiles_by_slide[s]) for s in train_slides)
    if len(val_slides) < max_val_slides and current_train >= target_train_tiles:
        val_slides.add(sid)
    else:
        train_slides.add(sid)

train_tiles_unique = [t for t in selected_tiles if t["slide_id"] in train_slides]
val_tiles          = [t for t in selected_tiles if t["slide_id"] in val_slides]

print(f"Slide quota: {len(train_slides)} train  /  {len(val_slides)} val  (target val ≤ {max_val_slides})")
print(f"TRAIN : {len(train_slides):2d} slides → {len(train_tiles_unique):,} tiles ({len(train_tiles_unique)/total_count*100:.1f}%)")
print(f"VAL   : {len(val_slides):2d} slides → {len(val_tiles):,} tiles ({len(val_tiles)/total_count*100:.1f}%)")
print(f"\nTrain slides : {sorted(train_slides)}")
print(f"Val slides   : {sorted(val_slides)}")

for cls_id in priority_ids:
    cname    = id_to_name[cls_id]
    in_train = any(cls_id in t["class_counts"] for t in train_tiles_unique)
    in_val   = any(cls_id in t["class_counts"] for t in val_tiles)
    if not in_train or not in_val:
        print(f"  WARNING: '{cname}' missing from {'train' if not in_train else 'val'}")

# ── Cap val to VAL_TARGET ─────────────────────────────────────────────────────
if len(val_tiles) > VAL_TARGET:
    np.random.seed(SEED + 2)
    idx = np.random.choice(len(val_tiles), size=VAL_TARGET, replace=False)
    val_tiles = [val_tiles[i] for i in idx]
    print(f"\nVal capped to {len(val_tiles):,} tiles (was {len([t for t in selected_tiles if t['slide_id'] in val_slides]):,})")
else:
    print(f"\nVal: {len(val_tiles):,} tiles (under VAL_TARGET={VAL_TARGET:,} — no cap needed)")

Slide quota: 19 train  /  3 val  (target val ≤ 3)
TRAIN : 19 slides → 28,014 tiles (80.0%)
VAL   :  3 slides → 6,986 tiles (20.0%)

Train slides : ['RM_GS33_0001', 'RM_GS33_0021', 'RM_GS33_0041', 'RM_GS33_0061', 'RM_GS33_0081', 'RM_GS33_0101', 'RM_GS33_0121', 'RM_GS33_0141', 'RM_GS33_0161', 'RM_GS33_0181', 'RM_GS33_0201', 'RM_GS33_0221', 'RM_GS33_0241', 'RM_GS33_0261', 'RM_GS33_0281', 'RM_GS33_0301', 'RM_GS33_0321', 'RM_GS33_0541', 'RM_GS33_0641']
Val slides   : ['RM_GS33_0341', 'RM_GS33_0361', 'RM_GS33_0431']

Val capped to 5,000 tiles (was 6,986)


---
## 9 · Oversample rare classes (train only)

> `OVERSAMPLE_TARGETS` below are tuned to GS55 values.
> Reduce targets for classes that have very few slides in GS33.

In [11]:
# ── Oversample rare / poorly-classified classes in training split ──────────────
MAX_AUG_FACTOR = 16

# *** Review these targets after seeing GS33 class counts from notebook 1 ***
OVERSAMPLE_TARGETS = {
    "lungs":      8000,
    "heart":      5000,
    "eye":        5000,
    "spleen":     5000,
    "pancreas":   4000,
    "kidney":     4000,
    "bladder":    3000,
    "spleen2":    3000,
    "thyroid":    3000,
    "ear":        3000,
    "mesokidney": 3000,
    "thymus":     2500,
}

train_by_cls = defaultdict(list)
for tile in train_tiles_unique:
    for cls_id in tile["class_counts"]:
        train_by_cls[cls_id].append(tile)

oversampled = []
np.random.seed(SEED + 1)

print(f"{'Class':<13} {'Have':>8} {'Target':>8} {'Added':>8}")
print("-" * 45)
for cls_name, tgt in OVERSAMPLE_TARGETS.items():
    cls_id    = name_to_id[cls_name]
    available = train_by_cls.get(cls_id, [])
    n_have    = len(available)
    if n_have == 0 or n_have >= tgt:
        print(f"  {cls_name:<11} {n_have:>8,} {tgt:>8,} {'—':>8}")
        continue
    tgt_capped = min(tgt, n_have * MAX_AUG_FACTOR)
    n_add      = tgt_capped - n_have
    chosen     = np.random.choice(n_have, size=n_add, replace=True)
    dup_ctr    = {}
    for orig_idx in chosen:
        k = dup_ctr.get(orig_idx, 0) + 1
        dup_ctr[orig_idx] = k
        dup = dict(available[orig_idx])
        dup["dup_suffix"] = f"_dup{k}"
        oversampled.append(dup)
    print(f"  {cls_name:<11} {n_have:>8,} {tgt_capped:>8,} {n_add:>8,}")

# ── Shuffle all train tiles and cap to exactly TRAIN_FINAL ────────────────────
combined = train_tiles_unique + oversampled
np.random.seed(SEED + 2)
np.random.shuffle(combined)
train_tiles = combined[:TRAIN_FINAL]

for t in train_tiles: t["split"] = "train"
for t in val_tiles:   t["split"] = "val"

n_unique_in_train = sum(1 for t in train_tiles if not t.get("dup_suffix"))
n_dup_in_train    = len(train_tiles) - n_unique_in_train
print(f"\nTrain pool before cap: {len(combined):,}  ({len(train_tiles_unique):,} unique + {len(oversampled):,} oversampled)")
print(f"Final: TRAIN {len(train_tiles):,}  ({n_unique_in_train:,} unique + {n_dup_in_train:,} augmented)  |  VAL {len(val_tiles):,}")

Class             Have   Target    Added
---------------------------------------------
  lungs             25      400      375
  heart            814    5,000    4,186
  eye              248    3,968    3,720
  spleen             6       96       90
  pancreas           6       96       90
  kidney           171    2,736    2,565
  bladder          230    3,000    2,770
  spleen2            0    3,000        —
  thyroid           68    1,088    1,020
  ear              298    3,000    2,702
  mesokidney         0    3,000        —
  thymus            14      224      210

Train pool before cap: 45,742  (28,014 unique + 17,728 oversampled)
Final: TRAIN 30,000  (18,318 unique + 11,682 augmented)  |  VAL 5,000


---
## 10 · Extract tiles — save PNGs and CSV labels

In [12]:
# ── Create output directory structure ─────────────────────────────────────────
for split in ("train", "val"):
    for sub in ("images", "geojsons", "labels"):
        (OUT_BASE / split / sub).mkdir(parents=True, exist_ok=True)

splits_dir = OUT_BASE / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)

(OUT_BASE / "label_map.json").write_text(
    json.dumps({str(k): v for k, v in id_to_name.items()}, indent=2), encoding="utf-8"
)
print(f"Output directory: {OUT_BASE}")

# ── Extract PNG tiles + GeoJSON labels + instance masks ───────────────────────
all_output = train_tiles + val_tiles
by_slide   = defaultdict(list)
for tile in all_output:
    by_slide[tile["slide_id"]].append(tile)

print(f"Processing {len(by_slide)} slides, {len(all_output):,} tiles total")

tile_records = []
failed       = []

for slide_id, slide_tiles in tqdm(by_slide.items(), desc="Slides"):
    cells_xy    = slide_data[slide_id]["cells_xy"]
    slide_feats = slide_data[slide_id]["features"]

    try:
        slide = openslide.OpenSlide(slide_data[slide_id]["image_path"])
    except Exception as exc:
        print(f"  ERROR opening {slide_id}: {exc}")
        failed += [(f"{slide_id}_{t['tile_x']}_{t['tile_y']}", str(exc)) for t in slide_tiles]
        continue

    for tile in slide_tiles:
        split      = tile["split"]
        dup_suffix = tile.get("dup_suffix", "")
        tile_id    = f"{slide_id}_x{tile['tile_x']}_y{tile['tile_y']}{dup_suffix}"
        m          = re.search(r"_dup(\d+)$", dup_suffix)
        aug_id     = (int(m.group(1)) % 8) if m else 0

        images_dir   = OUT_BASE / split / "images"
        geojsons_dir = OUT_BASE / split / "geojsons"
        labels_dir   = OUT_BASE / split / "labels"

        try:
            # 1. Read and augment RGB tile
            pil = slide.read_region((tile["tile_x"], tile["tile_y"]), 0, (TILE_SIZE, TILE_SIZE))
            img = np.array(pil.convert("RGB"))
            if img.shape[:2] != (TILE_SIZE, TILE_SIZE):
                img = cv2.resize(img, (TILE_SIZE, TILE_SIZE), interpolation=cv2.INTER_LINEAR)
            img = augment_image(img, aug_id)
            cv2.imwrite(
                str(images_dir / f"{tile_id}.png"),
                cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
            )

            # 2. Clip GeoJSON features to this tile and localize coordinates
            tile_feats = clip_features_to_tile(
                slide_feats,
                tile["tile_x"], tile["tile_y"], TILE_SIZE,
                tile["cell_indices"],
            )

            # 3. Augment polygon coordinates to match image augmentation
            if aug_id != 0:
                for feat in tile_feats:
                    rings = feat["geometry"]["coordinates"]
                    feat["geometry"]["coordinates"] = [
                        augment_polygon_ring(ring, aug_id, TILE_SIZE)
                        for ring in rings
                    ]

            # 4. Write per-tile GeoJSON (QuPath-compatible)
            write_tile_geojson(tile_feats, geojsons_dir / f"{tile_id}.geojson")

            # 5. Rasterize polygons → uint16 instance mask + inst2class.json
            mask, inst2class = rasterize_tile_features(tile_feats, TILE_SIZE)
            cv2.imwrite(str(labels_dir / f"{tile_id}.png"), mask)
            (labels_dir / f"{tile_id}_inst2class.json").write_text(
                json.dumps(inst2class), encoding="utf-8"
            )

            tile_records.append({
                "tile_id":      tile_id,
                "slide_id":     slide_id,
                "split":        split,
                "num_cells":    tile["num_cells"],
                "is_duplicate": bool(dup_suffix),
                "aug_id":       aug_id,
            })
        except Exception as exc:
            failed.append((tile_id, str(exc)))

    slide.close()

n_ok  = len(tile_records)
n_dup = sum(1 for r in tile_records if r["is_duplicate"])
print(f"\n[DONE] {n_ok:,} tiles saved  ({n_ok-n_dup:,} unique + {n_dup:,} augmented duplicates)")
if failed:
    print(f"Failed: {len(failed)}")
    for tid, err in failed[:5]:
        print(f"  {tid}: {err}")

Output directory: \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS33\cellvit_training\dataset_256_30k_GS33
Processing 22 slides, 35,000 tiles total


Slides:   0%|          | 0/22 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## 11 · Write split files and training config

In [ ]:
# ── Write split CSV files and training config ──────────────────────────────────
df_tiles  = pd.DataFrame(tile_records)

train_ids = df_tiles[df_tiles["split"] == "train"]["tile_id"].tolist()
val_ids   = df_tiles[df_tiles["split"] == "val"]["tile_id"].tolist()
fold_dir  = splits_dir / "fold_0"
fold_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(train_ids).to_csv(fold_dir / "train.csv", index=False, header=False)
pd.DataFrame(val_ids).to_csv(fold_dir / "val.csv",     index=False, header=False)

config = {
    "logging": {
        "mode": "online",
        "project": "convnext_stardist_GS33",
        "log_comment": "GS33_multitask_20x",
    },
    "data": {
        "train_images_dir": str(OUT_BASE / "train" / "images"),
        "train_labels_dir": str(OUT_BASE / "train" / "labels"),
        "val_images_dir":   str(OUT_BASE / "val"   / "images"),
        "val_labels_dir":   str(OUT_BASE / "val"   / "labels"),
        "train_stems":      str(fold_dir / "train.csv"),
        "val_stems":        str(fold_dir / "val.csv"),
        "num_classes":      len(LABELS),
        "label_map":        id_to_name,
    },
    "model": {
        "class_names": LABELS,
        "num_classes": len(LABELS),
        "n_rays": 32,
    },
    "train": {
        "patch_size":    TILE_SIZE,
        "batch_size":    8,
        "epochs":        50,
        "learning_rate": 0.0001,
        "loss_w_cls":    2.0,
        "loss_w_inst":   0.5,
        "loss_w_dist":   0.05,
        "class_weights": "auto",
    },
}

config_dir  = OUT_BASE / "train_configs"
config_dir.mkdir(parents=True, exist_ok=True)
config_path = config_dir / "train_config.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

(OUT_BASE / "label_map.yaml").write_text(
    yaml.safe_dump({"labels": id_to_name}), encoding="utf-8"
)

print(f"TRAIN: {len(train_ids):,} tiles  VAL: {len(val_ids):,}")
print(f"Splits → {splits_dir}")
print(f"Config → {config_path}")
print("\nDataset layout:")
for split in ("train", "val"):
    imgs  = list((OUT_BASE / split / "images").glob("*.png"))
    masks = list((OUT_BASE / split / "labels").glob("*.png"))
    print(f"  {split:5s}: {len(imgs):,} images  {len(masks):,} instance masks")
print("\nDataset creation complete!")

---
## 12 · Organ coverage summary

In [ ]:
# ── Organ coverage summary: tiles per class per split ─────────────────────────
from collections import defaultdict as _dd

split_cls_tiles = {"train": _dd(int), "val": _dd(int)}

for tile in train_tiles:
    for cls_id in tile["class_counts"]:
        split_cls_tiles["train"][id_to_name[cls_id]] += 1

for tile in val_tiles:
    for cls_id in tile["class_counts"]:
        split_cls_tiles["val"][id_to_name[cls_id]] += 1

print(f"{'Organ':<14} {'TRAIN':>8} {'VAL':>8}  Coverage")
print("-" * 45)
for name in LABELS:
    tr = split_cls_tiles["train"][name]
    va = split_cls_tiles["val"][name]
    flags = []
    if tr == 0: flags.append("NO TRAIN")
    if va == 0: flags.append("NO VAL")
    flag_str = "  WARNING: " + ", ".join(flags) if flags else ""
    print(f"{name:<14} {tr:>8,} {va:>8,}{flag_str}")

print(f"\n{'TOTAL':<14} {len(train_tiles):>8,} {len(val_tiles):>8,}")